# Anotador TDAH · 05 · Comparación de backends

Este cuaderno **no lanza ninguna llamada al modelo**: lee la tabla `anotacion` y cruza los resultados que dejaron los cuadernos 01–04 (y cualquier simulación anterior).

Pregunta que responde: **¿qué backend da la anotación más fiable, a qué coste?**

## 1 · Cargar las anotaciones

Por defecto carga la tabla entera. Si solo interesan las simulaciones de los cuadernos 01–04 de hoy, usar `cargar_df(desde=dt.datetime(2026, 7, 12))` con la fecha que toque, o filtrar el DataFrame por `modelo` / `temperature` para comparar en igualdad de condiciones.

In [ ]:
import datetime as dt

import matplotlib.pyplot as plt
import pandas as pd

from anotador.analisis import (
    cargar_df,
    krippendorff_nivel,
    krippendorff_por_semana,
    resumen_por_parametro,
)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

# DESDE = dt.datetime(2026, 7, 12)   # ← descomentar para acotar
DESDE = None

df = cargar_df(desde=DESDE)
print(f"{len(df)} anotaciones cargadas")
print(df.groupby(["backend", "modelo"]).size().rename("n"))

## 2 · Igualdad de condiciones

Para que la comparación sea justa, todos los backends deben compararse con el **mismo modelo y la misma temperatura**. Esta celda fija esa condición (ajustar a lo que se haya ejecutado).

In [ ]:
MODELO_COMPARACION = "gemma4:e4b"
TEMPERATURA_COMPARACION = 0.7

df_comp = df[(df["modelo"] == MODELO_COMPARACION)
             & (df["temperature"] == TEMPERATURA_COMPARACION)
             & (df["agregacion"] == "individual")].copy()
print(f"{len(df_comp)} anotaciones en condiciones comparables")
print(df_comp.groupby("backend").size().rename("n"))

## 3 · Tabla comparativa

Una fila por backend: consistencia (Jaccard, acuerdo de nivel), calidad media (0–5), fallo de formato y latencia.

In [ ]:
tabla = resumen_por_parametro(df_comp, "backend")
tabla = tabla.sort_values("jaccard_items", ascending=False)
display(tabla[["backend", "jaccard_items", "jaccard_escalas",
               "acuerdo_nivel", "media_metricas",
               "tasa_fallo_formato", "media_latencia"]])

## 4 · Alpha de Krippendorff por backend

Fiabilidad entre réplicas del nivel de alerta, una fila por configuración. Referencia: ≥ 0.80 fiable, 0.67–0.80 aceptable, < 0.67 insuficiente.

In [ ]:
alpha = krippendorff_nivel(df_comp)
display(alpha.sort_values("alpha_nivel", ascending=False)
        [["backend", "modelo", "temperature", "alpha_nivel"]])

## 5 · Evolución por semana

La fiabilidad de cada backend, semana a semana. Si un backend solo es bueno en algunas semanas (p. ej. textos más fáciles), aquí se ve.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

for backend, g in df_comp.dropna(subset=["semana"]).groupby("backend"):
    r = resumen_por_parametro(g, "semana")
    ax[0].plot(r["semana"], r["jaccard_items"], "o-", label=backend)
    ax[1].plot(r["semana"], r["media_latencia"], "o-", label=backend)

ax[0].set_xlabel("semana"); ax[0].set_ylabel("Jaccard ítems")
ax[0].set_ylim(0, 1.05); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[0].set_title("Consistencia por semana y backend")

ax[1].set_xlabel("semana"); ax[1].set_ylabel("latencia media (s)")
ax[1].legend(); ax[1].grid(alpha=0.3)
ax[1].set_title("Coste por semana y backend")

plt.tight_layout(); plt.show()

In [ ]:
alpha_sem = krippendorff_por_semana(df_comp)
if not alpha_sem.empty:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for backend, g in alpha_sem.groupby("backend"):
        ax.plot(g["semana"], g["alpha_nivel"], "o-", label=backend)
    ax.axhline(0.80, color="green", ls=":", lw=1, label="fiable (0.80)")
    ax.axhline(0.67, color="orange", ls=":", lw=1, label="aceptable (0.67)")
    ax.set_xlabel("semana"); ax.set_ylabel("alpha de Krippendorff")
    ax.set_ylim(0, 1.05); ax.legend(); ax.grid(alpha=0.3)
    ax.set_title("Fiabilidad del nivel de alerta por semana")
    plt.tight_layout(); plt.show()

## 6 · Conclusión

Para rellenar tras la comparación:

1. **Backend recomendado**: el de mayor fiabilidad (Jaccard + alpha) con fallo de formato ≈ 0.
2. **Compromiso fiabilidad/coste**: ¿el mejor backend justifica su latencia frente al segundo?
3. **Estabilidad temporal**: ¿la fiabilidad se mantiene entre semanas o hay semanas sistemáticamente peores? (Si las hay, mirar los textos de esas semanas: puede ser señal de reportes ambiguos.)
4. **Siguiente paso**: con el backend elegido, estudiar el efecto de temperatura y del consenso por voto mayoritario (`agregaciones=["consenso_voto"]` en la `Rejilla`).